# Recommender system: SVD and beyond

In [ ]:
import numpy as np
import pandas as pd

## Simulated movie rating data
- each row is a user
- each column is a movie
- some ratings are unknown 

**Goal: guess the ratings for unrated movies**

In [ ]:
R0 = np.full([8,11], fill_value=np.nan)
R0[:4,:]= np.array([[4.5,4.5,4.5,2,3.5,3,5,2.5,4.5,5,4],
                    [4,4,4.5,1.5,4,3.5,5,2,4.5,5,4],
                    [5,5,3.5,4,2.5,2,4,5,4.5,4,3.5],
                    [4,5,4,2.5,3,3,4.5,3,5,4.5,3.5]])
R0[4,1]=5;R0[4,9]=3;
R0[5,2]=5;R0[5,6]=4.5;
R0[6,3]=2;R0[6,10]=4.5;
R0[7,4]=3.5; R0[7,6]=2;
R0
R0_df = pd.DataFrame(R0, columns= ['E.T.',
                                   'Dracula', 
                                   'Die Hard', 
                                   'The Room',
                                   'Love, actually',
                                   'No Time to Die',
                                   'Lion King',
                                   'Top Gun',
                                   'Brave Heart',
                                   'Fifth Element',
                                   'True Romance',])
R0_df

## Create a mask to indicate known ratings
- known entry = 1
- unknown entry = 0

In [ ]:
mask = np.ones(R0.shape)
mask[np.isnan(R0)]=0
mask

## Main Idea
*Perception of the whole is based on perception of its parts.*

The ratings, collectively, has a low-dimensional structure. This means **the rating matrix is approximately low rank**.

Here is a very simplified example: we assume users are rating movies based on only **two features**

- User 0 decides how important each feature is to them: 

||Feature 1 (story) |Feature 2 (action)|
|:--|--|--|
|user 0|1.16 | 2.3|

- "General concensus" on how each movie scores on these features:

||E.T.|Dracula|
|:--|:--|--|
|Feature 1 (story)|1.54|1.67|
|Feature 2 (action)|1.08|1.12|

- Final rating
  - user 0's rating for *E.T.*: $\begin{bmatrix}1.16&2.3\end{bmatrix}\begin{bmatrix}
1.54\\
1.08
\end{bmatrix}=4.27$

$$R \approx \begin{bmatrix}1.16&2.3\\
?&?\\
?&?\\
?&?\\
?&?\\
?&?\\
?&?\\
?&?\\
?&?
\end{bmatrix}\begin{bmatrix}
1.54&&?&?&?&?&?&?&?&?&?\\
1.08&&?&?&?&?&?&?&?&?&?
\end{bmatrix}=PQ$$


## Initial simple imputation
This is often done by filling with averages of each movie (column)

In [ ]:
np.set_printoptions(precision = 2, suppress = True)

In [ ]:
# compute average (with NA's excluded)
col_avg = np.nanmean(R0, axis=0) # along axis 0 (rows), so col mean
R0_m = R0.copy()
for j in range(R0.shape[1]):
    R0_m[np.isnan(R0[:,j]),j] = col_avg[j]
print(R0_m)

## Method 1: plain SVD

In [ ]:
U, S, Vt = np.linalg.svd(R0_m, full_matrices=False)
V = Vt.T
k = 2 # can be adjusted
Sm = np.diag(S[:k])
Rk1 = U[:,:k]@Sm@(V[:,:k]).T # this is rank-k approximation of R0
for x, y in zip(Rk1, R0):
    print(f"{x} \t {y}")

In [ ]:
# What is P, Q here?

On the known ratings, Rk1 has different entries than R0

...

In [ ]:
def error_known(filled, given, mask):
    e = np.linalg.norm(filled*mask - given*mask, 'fro')
    e = e**2
    e = e/mask.sum()
    return(e**0.5)

In [ ]:
R0_SVD1 = Rk1.copy()
R0_SVD1[mask.astype(bool)] = R0[mask.astype(bool)]
R0_SVD1

In [ ]:
error = error_known(Rk1, R0_m, mask)
print(Rk1)
print(error)

## Method 2: SVD - with centering
$R\approx  PQ + $ col_avg


In [ ]:
col_avg = np.mean(R0_m, axis=0)
R0_fill_c = R0_m - col_avg
U,S,Vt = np.linalg.svd(R0_fill_c)
V = Vt.T
k = 2
Sm = np.diag(S[:k])
# Rk is the rank-k approximation of R0_m
Rk2 = U[:,:k]@Sm@(V[:,:k]).T+col_avg
error = error_known(Rk2, R0_m, mask)
print(Rk2)
print(error)

## Method 3: Nonnegative Matrix Factorization
Non-negative Matrix Factorization is to write a given matrix (with all positive entries) as product of two low-rank matrices, **both with nonnegative entries**:
$$R\approx PQ, \qquad P\in\R^{m\times k}, Q\in\R^{k\times n}, P_{ij}\geq0, Q_{ij}\geq0$$
$P$ and $Q$ are **low-rank** as $k\ll m, k\ll n$ 

Applications:
- originated in chemometrics
- genomics
- computer vision
- natural language processing (topic modeling)
- recommender system
- ...


In [ ]:
max_iter = 350
m, n = R0_m.shape
R = R0_m*mask
np.random.seed(3)
P = np.random.rand(m, k)
Q = np.random.rand(k, n)
eta = 0.01
for i in range(max_iter):
    PQ = (P@Q)*mask
    P = P + eta*(R@Q.T - PQ@Q.T)
    Q = Q + eta*(P.T@R - P.T@PQ)

In [ ]:
R_nmf = P@Q
error = error_known(R_nmf, R0_m, mask)
print(error)